# Fire Door Detection — Model Training (Google Colab)

Trains the **3 trainable models** on Colab's free GPU, then exports weights to drop into the FastAPI backend (`backend/models/`).

| # | Model | Ultralytics base | Output weight |
|---|---|---|---|
| 1 | Door symbol detection | `yolo11n.pt` (detect) | `door_yolo11.pt` |
| 2 | Door-type classification | `yolo11n-cls.pt` (classify) | `door_type.pt` |
| 5 | Fire-barrier / wall-rating | `yolo11n-seg.pt` (segment) | `wall_seg.pt` |

**Before you start:** Runtime → Change runtime type → **GPU (T4)**.

Workflow: run each section → the final section zips all weights → download → unzip into `backend/models/`.

## 0. Setup — verify GPU and install

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

In [ ]:
%pip -q install ultralytics roboflow
from ultralytics import YOLO
import ultralytics; ultralytics.checks()

In [ ]:
# Where trained weights get collected before download.
from pathlib import Path
WEIGHTS = Path('/content/weights'); WEIGHTS.mkdir(exist_ok=True)
print('weights ->', WEIGHTS)

## 1. Door symbol detection (YOLOv11 detect) — component #1

### Get your free Roboflow key (~30s)
1. Sign up at [roboflow.com](https://roboflow.com) (free).
2. **Settings → API →** copy your *Private API Key*.
3. Paste it into `ROBOFLOW_API_KEY` below.

### Recommended floor-plan door datasets (open on Universe, pick one)
| Dataset | Notes |
|---|---|
| [plandoor/doorplan](https://universe.roboflow.com/plandoor/doorplan) | 2,308 imgs, doors on floor plans, ~90% mAP — **best fit** |
| [gilbertyolov5/yolo-door-detection](https://universe.roboflow.com/gilbertyolov5/yolo-door-detection) | door symbols |
| [keerthi-edrbd/detecting-doors-from-floor-plan](https://universe.roboflow.com/keerthi-edrbd/detecting-doors-from-floor-plan) | doors from floor plans |

### Easiest: paste Roboflow's exact snippet
On the dataset page click **Download Dataset → format `YOLOv11` → show download code**.
Roboflow gives you a `rf.workspace(...).project(...).version(N).download('yolov11')`
line with the correct workspace/project/version. Paste that over the marked line
in the next cell. (Pre-filled values are a sensible default, but dataset versions
change — the copied snippet is always correct.)

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''   # <-- paste your Private API Key (Settings -> API)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# ---- PASTE Roboflow's exact download line here (Download Dataset -> YOLOv11) ----
# It looks like: rf.workspace("plandoor").project("doorplan").version(N)
project = rf.workspace("plandoor").project("doorplan")
version = project.version(1)          # <-- match the version Roboflow shows
# -------------------------------------------------------------------------------

ds = version.download("yolov11")
DATA_YAML = ds.location + "/data.yaml"
print("data.yaml ->", DATA_YAML)
print(open(DATA_YAML).read())         # sanity-check classes (expect a 'door' class)

In [ ]:
# Train. Bump epochs once you confirm it runs; 50 is a solid baseline.
model = YOLO('yolo11n.pt')
model.train(data=DATA_YAML, epochs=50, imgsz=1024, batch=8,
            project='runs', name='door_detect')

In [ ]:
# Validate + quick visual sanity check on a val image.
metrics = model.val()
print('mAP50-95:', metrics.box.map, '| mAP50:', metrics.box.map50)
import glob
sample = glob.glob(ds.location + '/valid/images/*')[:1]
if sample:
    model.predict(sample, save=True, imgsz=1024)

In [ ]:
import shutil
# model.trainer.best is the exact path YOLO saved best.pt to (folder can vary).
best = model.trainer.best
shutil.copy(best, WEIGHTS / 'door_yolo11.pt')
print('saved door_yolo11.pt from', best)

## 2. Door-type classification (YOLOv11 classify) — component #2

Classes: `single`, `double`, `fire` (extend as needed). Needs an **image
classification** dataset (folder-per-class). Export from Roboflow as
`folder`, or crop door boxes from Section 1 into `train/<class>/` and
`val/<class>/`. Set `CLS_DATA` to that folder root.

In [ ]:
# Option A: Roboflow classification project exported as 'folder'
# ds_cls = rf.workspace('WS').project('door-type').version(1).download('folder')
# CLS_DATA = ds_cls.location

# Option B: your own folder tree (train/<class>/*.png, val/<class>/*.png)
CLS_DATA = '/content/door_type_dataset'   # <-- set this

cls = YOLO('yolo11n-cls.pt')
cls.train(data=CLS_DATA, epochs=40, imgsz=224, project='runs', name='door_type')

In [ ]:
shutil.copy(cls.trainer.best, WEIGHTS / 'door_type.pt')
print('saved door_type.pt')

## 3. Fire-barrier / wall-rating segmentation (YOLOv11 segment) — component #5

Fire-rated walls are drawn as distinct line patterns. This needs a
**segmentation** dataset (polygon masks of rated wall runs), which you must
label yourself (Roboflow → Instance Segmentation → export `yolov11`).
Until you have it, this section stays a scaffold — it trains the moment you
point `SEG_DATA` at a real `data.yaml`.

In [ ]:
# ds_seg = rf.workspace('WS').project('wall-rating-seg').version(1).download('yolov11')
# SEG_DATA = ds_seg.location + '/data.yaml'
SEG_DATA = None   # <-- set to your seg data.yaml when ready

if SEG_DATA:
    seg = YOLO('yolo11n-seg.pt')
    seg.train(data=SEG_DATA, epochs=60, imgsz=1024, batch=8,
              project='runs', name='wall_seg')
    shutil.copy(seg.trainer.best, WEIGHTS / 'wall_seg.pt')
    print('saved wall_seg.pt')
else:
    print('SEG_DATA not set — skipping (label a segmentation dataset first).')

## 4. Download weights → backend/models/

Downloads a `fire_door_weights.zip`. Unzip its contents into
`backend/models/` so the pipeline loaders find `door_yolo11.pt`,
`door_type.pt`, `wall_seg.pt`.

In [ ]:
import shutil
shutil.make_archive('/content/fire_door_weights', 'zip', WEIGHTS)
print('contents:', [p.name for p in WEIGHTS.iterdir()])
try:
    from google.colab import files
    files.download('/content/fire_door_weights.zip')
except Exception as e:
    print('Manual download from Files panel:', e)